<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_35_drf_fastapi/note_lesson_35_drf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔌 Урок 35 — Django REST Framework: API до нотаток

| Крок | Що робимо |
|---|---|
| 0 | запускаємо `hello_project` з API у ноутбуці |
| 1 | серіалізація: модель → JSON (вправи 1–2) |
| 2 | валідація і збереження (вправи 3–4) |
| 3 | API через `APIClient`: список, сторінки (вправа 5) |
| 4 | права: читати всім, писати після входу (вправа 6) |
| 5 | фільтри і дія `pin` (вправи 7–8) |
| 6 | знайди помилку: `fields = "__all__"` (вправа 9) |
| 7 | той самий API на FastAPI |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія — у книзі курсу: [Урок 35](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/).

---
## 0. Проєкт у ноутбуці

Проєкт — `hello_project` з уроку 33 + DRF: `hello_app/serializers.py`, `hello_app/api.py`, роутер у `hello_project/urls.py`. У Colab клітинка нижче завантажить лише папку уроку і встановить пакети.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_35_drf_fastapi"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("hello_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

packages = {"django": "Django>=5.2,<6", "rest_framework": "djangorestframework", "drf_spectacular": "drf-spectacular",
            "fastapi": "fastapi", "httpx": "httpx"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

os.chdir("hello_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (див. урок 33)
django.setup()

from django.contrib.auth.models import User
from django.test.utils import setup_test_environment
from rest_framework.test import APIClient

from hello_app.models import Note
from hello_app.serializers import NoteSerializer

setup_test_environment()                               # хост testserver для тестового клієнта
Note.objects.all().delete()
Note.objects.create(title="Купити молоко", content="2 л")
Note.objects.create(title="Вивчити DRF", content="serializers, viewsets", priority=3)
Note.objects.create(title="Пароль від Wi-Fi", content="coffee2026", is_pinned=True)
Note.objects.create(title="Ідея: бот для нотаток", priority=1)
student, _ = User.objects.get_or_create(username="student")
print("нотаток:", Note.objects.count())

---
## 1. Серіалізація

`NoteSerializer(об'єкт).data` — словник для JSON; `many=True` — для колекції.

### Вправа 1

Серіалізуй нотатку «Вивчити DRF» у `data`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
data = NoteSerializer(Note.objects.get(title="Вивчити DRF")).data
# END SOLUTION

assert set(data) == {"id", "title", "content", "is_pinned", "priority", "priority_label", "created_at"}
assert (data["priority"], data["priority_label"]) == (3, "Високий")
print(data)
print("✅ Вправа 1 пройдена")

### Вправа 2

`titles` — заголовки всіх **незакріплених** нотаток із серіалізованого списку (`many=True`), у порядку моделі.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
rows = NoteSerializer(Note.objects.filter(is_pinned=False), many=True).data
titles = [row["title"] for row in rows]
# END SOLUTION

assert titles == ["Ідея: бот для нотаток", "Вивчити DRF", "Купити молоко"], titles
print("✅ Вправа 2 пройдена")

---
## 2. Валідація і збереження

Спершу `serializer.is_valid()`, потім `serializer.save()`. Помилки — у `serializer.errors`.

### Вправа 3

Перевір дані `{"title": "ok", "priority": 7}`: збережи результат `is_valid()` у `ok` і **назви полів** з помилками (множиною) у `bad_fields`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
serializer = NoteSerializer(data={"title": "ok", "priority": 7})
ok = serializer.is_valid()
bad_fields = set(serializer.errors)
# END SOLUTION

assert ok is False
assert bad_fields == {"title", "priority"}
print(serializer.errors)
print("✅ Вправа 3 пройдена")

### Вправа 4

Створи через серіалізатор нотатку з даних `{"title": "  Здати звіт  ", "priority": 4, "id": 777}`. Збережений об'єкт — у `created`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
serializer = NoteSerializer(data={"title": "  Здати звіт  ", "priority": 4, "id": 777})
serializer.is_valid(raise_exception=True)
created = serializer.save()
# END SOLUTION

assert created.title == "Здати звіт", "пробіли прибирає validate_title"
assert created.id != 777, "id — лише для читання"
assert created.get_priority_display() == "Терміновий"
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** Що станеться, якщо прибрати `"id"` з `read_only_fields` (і з `fields` теж)?

<details>
<summary>Відповідь</summary>

Якщо прибрати з `read_only_fields`, ModelSerializer однаково зробить автоматичне поле `id` лише для читання — таке поле клієнт не задає. Якщо прибрати `id` з `fields`, його не буде у відповіді, і клієнт не знатиме адресу нотатки.

</details>

---
## 3. API через тестовий клієнт

`APIClient` надсилає запити до роутера DRF без мережі — як `curl` до `runserver`.

### Вправа 5

Отримай першу сторінку `GET /api/notes/` у `page1`, другу (за посиланням `next`) — у `page2`. Тіло відповіді — `response.json()`.

In [ ]:
client = APIClient()
# YOUR CODE HERE
# BEGIN SOLUTION
page1 = client.get("/api/notes/").json()
page2 = client.get(page1["next"]).json()
# END SOLUTION

assert page1["count"] == 5 and len(page1["results"]) == 3
assert page1["results"][0]["title"] == "Пароль від Wi-Fi", "закріплені — першими"
assert len(page2["results"]) == 2 and page2["next"] is None
print("✅ Вправа 5 пройдена")

---
## 4. Права

`IsAuthenticatedOrReadOnly`: читати всім, змінювати — після входу. У тестах вхід — `client.force_authenticate(user)`.

### Вправа 6

Спробуй `POST /api/notes/` з `{"title": "Нова нотатка"}` **без входу** (код — у `anonymous`), потім увійди як `student` і повтори (відповідь — у `created`). Не забудь `format="json"`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
anonymous = client.post("/api/notes/", {"title": "Нова нотатка"}, format="json").status_code
client.force_authenticate(student)
created = client.post("/api/notes/", {"title": "Нова нотатка"}, format="json")
# END SOLUTION

assert anonymous == 403
assert created.status_code == 201 and created.json()["priority_label"] == "Звичайний"
print("✅ Вправа 6 пройдена")

**🔮 Прогноз:** Чому `403`, а не `401`?

<details>
<summary>Відповідь</summary>

Перший клас автентифікації в `REST_FRAMEWORK` — сесійний, він не вміє попросити облікові дані заголовком `WWW-Authenticate`. Якби першим стояв `BasicAuthentication`, DRF повернув би `401`.

</details>

---
## 5. Фільтри і дія pin

`NoteViewSet.get_queryset()` читає `?pinned=` і `?search=`; `POST /api/notes/{id}/pin/` закріплює нотатку.

### Вправа 7

`pinned_titles` — заголовки з `GET /api/notes/?pinned=true`; `found` — кількість (`count`) для `?search=drf`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
pinned_titles = [n["title"] for n in client.get("/api/notes/", {"pinned": "true"}).json()["results"]]
found = client.get("/api/notes/", {"search": "drf"}).json()["count"]
# END SOLUTION

assert pinned_titles == ["Пароль від Wi-Fi"]
assert found == 1
print("✅ Вправа 7 пройдена")

### Вправа 8

Закріпи нотатку «Купити молоко» через дію `pin` (клієнт уже увійшов як `student`). Відповідь — у `response`.

In [ ]:
milk = Note.objects.get(title="Купити молоко")
# YOUR CODE HERE
# BEGIN SOLUTION
response = client.post(f"/api/notes/{milk.id}/pin/")
# END SOLUTION

assert response.status_code == 200 and response.json()["is_pinned"] is True
milk.refresh_from_db()
assert milk.is_pinned
print("✅ Вправа 8 пройдена")

---
## 6. Знайди помилку

```python
class UserSerializer(serializers.ModelSerializer):
    class Meta:
        model = User
        fields = "__all__"
```

Такий серіалізатор віддає хеш пароля і `is_superuser`.

### Вправа 9

Напиши `SafeUserSerializer` з **явним** списком полів: `id`, `username`, `first_name`, `last_name`.

In [ ]:
from rest_framework import serializers

# YOUR CODE HERE
# BEGIN SOLUTION
class SafeUserSerializer(serializers.ModelSerializer):
    class Meta:
        model = User
        fields = ["id", "username", "first_name", "last_name"]
# END SOLUTION

data = SafeUserSerializer(student).data
assert set(data) == {"id", "username", "first_name", "last_name"}
assert "password" not in data and "is_superuser" not in data
print(data)
print("✅ Вправа 9 пройдена")

---
## 7. Той самий API на FastAPI

Файл `fastapi_notes.py` у проєкті — ті самі ендпоінти на FastAPI з Pydantic. Порівняй відповідь на неправильні дані: DRF — `400` і помилки за полями, FastAPI — `422` і список `detail`.

In [ ]:
import warnings

warnings.filterwarnings("ignore")                      # попередження нових версій starlette про httpx
from fastapi.testclient import TestClient

from fastapi_notes import app

fastapi_client = TestClient(app)
print(fastapi_client.post("/api/notes/", json={"title": "Вивчити FastAPI"}).status_code)
bad = fastapi_client.post("/api/notes/", json={"title": "ok", "priority": 9})
print(bad.status_code, [(error["loc"][-1], error["msg"]) for error in bad.json()["detail"]])
drf_bad = client.post("/api/notes/", {"title": "ok", "priority": 9}, format="json")
print(drf_bad.status_code, drf_bad.json())

---
## Самоперевірка

1. Які дві роботи виконує серіалізатор?
2. Навіщо `read_only_fields`?
3. Які адреси дає `router.register("notes", NoteViewSet)`?
4. Навіщо `get_queryset()`, якщо є атрибут `queryset`?
5. Чому `fields = "__all__"` небезпечний?

<details>
<summary>Відповіді</summary>

1. Об'єкт → JSON (серіалізація) і JSON → перевірені дані → модель (десеріалізація з валідацією).
2. Щоб клієнт не міг задати поля, які визначає сервер (`id`, час створення).
3. `GET/POST /api/notes/`, `GET/PUT/PATCH/DELETE /api/notes/{id}/`, корінь `/api/`, адреси дій `@action`.
4. Фільтри залежать від параметрів кожного запиту; `get_queryset()` викликається на кожен запит.
5. Відкриває всі колонки (хеш пароля, `is_superuser`) і дозволяє їх змінювати.

</details>

## Далі

- Книга курсу: [Урок 35](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/) — налаштування DRF, `curl` до живого сервера, browsable API, OpenAPI, тести, порівняння з FastAPI.
- Запусти в терміналі: `python manage.py runserver` у папці `hello_project` і відкрий `http://127.0.0.1:8000/api/notes/` у браузері.
- **Урок 36** — типізація і Pydantic — основа FastAPI.